# B1 · 凌日光曲線的貝葉斯參數估計 —— 我們找得到那顆行星嗎？

> **核心問題**：一顆行星經過恆星前面，亮度掉了約 0.02%。**從這個微小的凹陷，我能推論出行星多大、軌道多長嗎？而且我對這些數字有多確定？**

這是唯一「你的作品流程 = 真實科學研究流程」的領域：下載 Kepler 光曲線 → 去趨勢 → 相位摺疊 → batman 物理模型 → **emcee**（天文標準）MCMC → 和已發表值對答案。目標 **Kepler-10b**。
核心程式在 [`../src/`](../src)。

In [1]:
import sys, os, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, arviz as az
import data, transit_model as tm, inference as inf
DATA = os.path.abspath('../../data/B_astro')
d = data.prepare(DATA)
P = float(d['P']); pub = data.published_kepler10b()
bp, bf, be, bn = data.bin_fold(d['fold_phase'], d['fold_flux'], window=0.06, n_bins=80)
print(f"總點數={int(d['n_points'])}  BLS 週期={P:.6f} d（已發表 {pub['P']:.6f}）")

總點數=51768  BLS 週期=0.837487 d（已發表 0.837491）


## 1 · 資料管線：下載 → 去趨勢 → 摺疊

`lightkurve` 抓全部 15 個季度 → stitch → 去 NaN/離群 → **flatten**（去恆星/儀器趨勢）→ BLS 找週期 → 遮罩凌日再 flatten（避免凹陷被稀釋）。

![去趨勢](../figures/01_detrend.png)

> **陷阱**：一定要先 detrend，否則恆星自轉/儀器漂移會蓋掉 0.02% 的凌日訊號。

## 2 · 相位摺疊：從幾千次凌日提取訊號

Kepler-10b 週期只有 20 小時 → 4 年資料有**幾千次凌日**。用已知週期把它們疊起來，訊噪比大幅提升。

![相位摺疊](../figures/02_phase_fold.png)

單次折疊的 cadence（灰）雜訊高達 ±500 ppm，但摺疊分箱後（藍），一個約 **195 ppm** 的凹陷清晰浮現。

## 3 · 物理模型（batman）與先驗設計

待估：`Rp/R*`（深度）、`a/R*`、`b`（撞擊參數）、`t0`、Kipping 臨邊昏暗 `q1,q2`、基線、jitter。先驗（步驟 4，要動腦）：
- `Rp/R*` **正數**（半徑比）
- `b` 均勻 $=\cos i$ 均勻（**幾何先驗**，不是角度均勻 ← 參數化不變性）
- 臨邊昏暗用 **Kipping (2013)** $(q_1,q_2)$（物理有效三角形），並用恆星模型理論值當先驗
- **對長曝光積分**（Kepler 長曝光 29.4 分鐘會抹平凌日邊緣——不積分會偏低 Rp）

In [2]:
ev = tm.TransitEvaluator(bp, P)   # 與 run_all 相同設定（64 walkers × 30000 步）
sampler, idata, tau = inf.run_emcee(ev, bf, be, seed=42)
fs = inf.flat_samples(sampler)
summ = az.summary(idata, var_names=inf.LABELS)
print(summ[['mean','sd','hdi_3%','hdi_97%','r_hat','ess_bulk']].to_string())

          mean     sd  hdi_3%  hdi_97%  r_hat  ess_bulk
rp       0.013  0.000   0.013    0.014   1.02    2974.0
a        3.999  0.438   3.087    4.514   1.02    2786.0
b        0.351  0.220   0.000    0.726   1.02    2647.0
t0       0.001  0.000   0.001    0.002   1.00   19987.0
q1       0.388  0.082   0.233    0.542   1.00   17870.0
q2       0.226  0.078   0.080    0.374   1.00   15034.0
f0       1.000  0.000   1.000    1.000   1.00   20452.0
log_jit -5.138  0.060  -5.253   -5.025   1.00   18961.0


## 4 · 結果：我們找到那顆行星了嗎？

![與已發表對照](../figures/05_published_comparison.png)

In [3]:
rp = fs[:,0]; a = fs[:,1]; b = fs[:,2]
inc = np.degrees(np.arccos(np.clip(b/a, 0, 1)))
rp_earth = rp * 1.065 * 109.1   # Rp = (Rp/R*)·R*，R*≈1.065 Rsun
lo,hi = np.percentile(rp,[2.5,97.5]); loe,hie = np.percentile(rp_earth,[2.5,97.5])
print(f"Rp/R* = {np.median(rp):.4f} [{lo:.4f}, {hi:.4f}]   已發表 {pub['rp_rs']}±0.0004")
print(f"Rp    = {np.median(rp_earth):.2f} R⊕ [{loe:.2f}, {hie:.2f}]   已發表 {pub['rp_earth']} → 涵蓋 {'✓' if loe<=pub['rp_earth']<=hie else '✗'}")
print(f"傾角 i = {np.median(inc):.1f}°   BLS 週期對上已發表到 {abs(P-pub['P'])*1e6:.0f} ppm")

Rp/R* = 0.0129 [0.0126, 0.0139]   已發表 0.01247±0.0004
Rp    = 1.50 R⊕ [1.46, 1.62]   已發表 1.47 → 涵蓋 ✓
傾角 i = 85.6°   BLS 週期對上已發表到 4 ppm


> **通關**：物理半徑 **Rp = 1.50 R⊕ 涵蓋已發表 1.47**；Rp/R* 與已發表在互相誤差內一致；BLS 週期對上已發表到百萬分之幾。**我們從 0.02% 的凹陷，重現了一顆已發表的行星。**

## 重點

1. 相位摺疊把幾千次凌日的訊號疊出來——0.02% 的凹陷從雜訊中現形。
2. batman + 貝葉斯給的不是點估計，是帶不確定性的行星半徑、傾角。
3. 和已發表值對答案：物理半徑涵蓋、週期吻合——這是作品最有力的驗證。

→ 但這些參數彼此**強烈簡併**，而且 MCMC 是唯一能誠實捕捉它的工具。見 [`02_degeneracy_and_diagnostics.ipynb`](02_degeneracy_and_diagnostics.ipynb)。